# Used Car Price Prediction
**DSC 148 – Intro to Data Mining | Course Project**  
Liuxuhong Zhao · Kangxin Peng  
University of California, San Diego

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import time
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

## 2. Load Data

In [ ]:
# Load dataset (place vehicles.csv in the same directory)
df = pd.read_csv('vehicles.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nBasic Stats:\n", df.describe())

## 3. Data Cleaning

In [ ]:
# Drop irrelevant columns
drop_cols = ['id', 'url', 'region_url', 'image_url', 'description', 'VIN', 'county']
df = df.drop(columns=drop_cols)

# Filter outliers
df = df[(df['price'] >= 500) & (df['price'] <= 150000)]
df = df[(df['year'] >= 1990) & (df['year'] <= 2023)]
df = df[(df['odometer'] >= 1000) & (df['odometer'] <= 400000)]

# Drop rows with missing critical fields
df = df.dropna(subset=['price', 'year', 'odometer', 'manufacturer', 'model'])

# Impute categorical columns with mode
for col in ['condition', 'cylinders', 'fuel', 'title_status', 'transmission',
            'drive', 'size', 'type', 'paint_color', 'state']:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Cleaned shape:", df.shape)
print("\nMissing after cleaning:\n", df.isnull().sum())
print("\nPrice stats:\n", df['price'].describe())

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price distribution
axes[0, 0].hist(df['price'], bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].set_xlabel('Price ($)')

# Car age distribution
df['car_age'] = 2024 - df['year']
axes[0, 1].hist(df['car_age'], bins=30, color='coral', edgecolor='white')
axes[0, 1].set_title('Car Age Distribution')
axes[0, 1].set_xlabel('Age (years)')

# Odometer distribution
axes[1, 0].hist(df['odometer'], bins=50, color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Odometer Distribution')
axes[1, 0].set_xlabel('Miles')

# Price vs car age
axes[1, 1].scatter(df['car_age'], df['price'], alpha=0.05, color='purple', s=1)
axes[1, 1].set_title('Price vs Car Age')
axes[1, 1].set_xlabel('Car Age')
axes[1, 1].set_ylabel('Price ($)')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150)
plt.show()

## 5. Feature Engineering

In [ ]:
# car_age already created above

# Extract posting month
df['posting_date'] = pd.to_datetime(df['posting_date'], utc=True, errors='coerce')
df['posting_month'] = df['posting_date'].dt.month.fillna(6).astype(int)

# Cylinders: extract number
df['cylinders_num'] = df['cylinders'].str.extract(r'(\d+)').astype(float)
df['cylinders_num'] = df['cylinders_num'].fillna(df['cylinders_num'].median())

# Target encoding: manufacturer and state
mfr_mean   = df.groupby('manufacturer')['price'].mean()
state_mean = df.groupby('state')['price'].mean()
df['manufacturer_enc'] = df['manufacturer'].map(mfr_mean)
df['state_enc']        = df['state'].map(state_mean)

# Label encode categorical columns
cat_cols = ['condition', 'fuel', 'title_status', 'transmission',
            'drive', 'size', 'type', 'paint_color']
le = LabelEncoder()
for col in cat_cols:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# Final feature list
feature_cols = [
    'car_age', 'odometer', 'cylinders_num',
    'manufacturer_enc', 'state_enc', 'posting_month',
    'condition_enc', 'fuel_enc', 'title_status_enc',
    'transmission_enc', 'drive_enc', 'size_enc',
    'type_enc', 'paint_color_enc'
]

X = df[feature_cols]
y = df['price']

print("Feature matrix shape:", X.shape)
print("Any NaN in X?", X.isnull().sum().sum())

## 6. Model Training & Evaluation

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

results = {}

# Linear Regression
print("\nTraining Linear Regression...")
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
results['Linear Regression'] = {
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'R2':  r2_score(y_test, y_pred_lr),
    'Time': round(time.time() - t0, 2)
}
print("Done:", results['Linear Regression'])

# Random Forest
print("\nTraining Random Forest...")
t0 = time.time()
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results['Random Forest'] = {
    'MAE': mean_absolute_error(y_test, y_pred_rf),
    'R2':  r2_score(y_test, y_pred_rf),
    'Time': round(time.time() - t0, 2)
}
print("Done:", results['Random Forest'])

# XGBoost
print("\nTraining XGBoost...")
t0 = time.time()
xgb = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=7,
                   random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
results['XGBoost'] = {
    'MAE': mean_absolute_error(y_test, y_pred_xgb),
    'R2':  r2_score(y_test, y_pred_xgb),
    'Time': round(time.time() - t0, 2)
}
print("Done:", results['XGBoost'])

print("\n========== Results ==========")
for model, metrics in results.items():
    print(f"{model}: MAE=${metrics['MAE']:.2f}, R2={metrics['R2']:.4f}, Time={metrics['Time']}s")

## 7. Feature Importance & Ablation Study

In [ ]:
# Feature importance plot
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

# Ablation study
print("\nAblation Study:")
baseline_mae = mean_absolute_error(y_test, xgb.predict(X_test))
print(f"Baseline MAE (all features): ${baseline_mae:.2f}")

for col in feature_cols:
    X_ablated = X_test.copy()
    X_ablated[col] = X_test[col].mean()
    ablated_mae = mean_absolute_error(y_test, xgb.predict(X_ablated))
    print(f"Without {col}: MAE=${ablated_mae:.2f} (Δ+${ablated_mae - baseline_mae:.2f})")